In [1]:
from tuya_connector import TuyaOpenAPI

ACCESS_ID = "jpdvgtxmhu5tsr7gcywj"
ACCESS_KEY = "0257343b7c724997a63f6e6a23f48d27"
API_ENDPOINT = "https://openapi.tuyaeu.com"
DEVICE_ID = "bf0178074ad9c548aduc79"

openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
openapi.connect()

# Get all status data
response = openapi.get(f"/v1.0/devices/{DEVICE_ID}/status")
print(response)

{'code': 28841002, 'msg': 'No permissions. Your subscription to cloud development plan has expired.', 'success': False, 't': 1771007227312, 'tid': '9a43d9e3090911f1a0e39aa9e01ed45b'}


In [ ]:
import base64
import time
import csv
from tuya_connector import TuyaOpenAPI

# Tuya IoT credentials
ACCESS_ID = "jpdvgtxmhu5tsr7gcywj"
ACCESS_KEY = "0257343b7c724997a63f6e6a23f48d27"
API_ENDPOINT = "https://openapi.tuyaeu.com"
DEVICE_ID = "bf0178074ad9c548aduc79"

# Initialize API
openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
openapi.connect()

CSV_FILE = "project_data.csv"

# Create CSV file with header
with open(CSV_FILE, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        "Time",
        "Voltage(V)",
        "Current(A)",
        "Power(W)",
        "Leakage Current(mA)",
        "Total Energy(kWh)",
        "Price(DZD)"
    ])

# Electricity tariff (price per kWh in Algeria)
TARIF_DZD = 5.34  

def get_measurements():
    voltage = current = power = energy = leakage = None
    response = openapi.get(f"/v1.0/devices/{DEVICE_ID}/status")

    for item in response.get("result", []):
        code = item["code"]

        # Phase A decoding for V/I/P
        if code == "phase_a":
            raw_bytes = base64.b64decode(item["value"])
            voltage_raw = raw_bytes[2] | (raw_bytes[3] << 8)
            current_raw = raw_bytes[4] | (raw_bytes[5] << 8)
            power_raw   = raw_bytes[6] | (raw_bytes[7] << 8)

            voltage = voltage_raw / 10.0
            current = current_raw / 100.0
            power   = power_raw / 10.0

        # Total energy (kWh)
        elif code == "total_forward_energy":
            energy = item["value"] / 100.0  

        # Leakage current
        elif code == "leakage_current":
            leakage = item["value"]

    return voltage, current, power, leakage, energy

# Collect and save data every 10 seconds
while True:
    voltage, current, power, leakage, energy = get_measurements()
    now = time.strftime("%Y-%m-%d %H:%M:%S")

    if voltage is not None and energy is not None:
        price = energy * TARIF_DZD  # 💡 Price = Energy × Tariff

        print(f"[{now}] V={voltage:.1f}V | I={current:.2f}A | "
            f"P={power:.1f}W | L={leakage}mA | "
            f"E={energy:.2f}kWh | Price={price:.2f} DZD")

        with open(CSV_FILE, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([now, voltage, current, power, leakage, energy, price])
    else:
        print(f"[{now}] No data found.")

    time.sleep(10)

[2026-02-13 19:27:14] No data found.
[2026-02-13 19:27:24] No data found.


KeyboardInterrupt: 

: 